# Lakehouse Data Engineering Workflow
This notebook demonstrates mounting storage, reading raw data, applying transformations and cleaning logic, and writing to bronze, silver, and gold layers.

In [ ]:
# Mount storage containers (replace placeholders)
dbutils.fs.mount(
  source = "wasbs://raw@<storage_account>.blob.core.windows.net/",
  mount_point = "/mnt/raw",
  extra_configs = {"<conf_key>": "<conf_value>"}
)
# Repeat for bronze, silver, gold

In [ ]:
# Read raw data (example: CSV)
raw_df = spark.read.csv('/mnt/raw/orders.csv', header=True, inferSchema=True)
raw_df.show(5)

In [ ]:
# Bronze: Basic normalization (Delta table)
bronze_df = raw_df.dropDuplicates()
bronze_df = bronze_df.withColumnRenamed('OrderID', 'order_id')
bronze_df.write.format('delta').mode('overwrite').save('/mnt/bronze/orders_delta')

In [ ]:
# Silver: Cleaning and business rules (Delta table)
from pyspark.sql.functions import col
silver_df = bronze_df.filter(col('order_id').isNotNull())
silver_df = silver_df.fillna({'quantity': 0})
silver_df.write.format('delta').mode('overwrite').save('/mnt/silver/orders_delta')

In [ ]:
# Gold: Aggregation for analytics (Delta table)
gold_df = silver_df.groupBy('product_id').sum('quantity')
gold_df.write.format('delta').mode('overwrite').save('/mnt/gold/product_sales_delta')